# Week 2 Day 1 - 4-Way Autism Conversation (Steph Moyo)

This notebook follows the Week 2 Day 1 conversation pattern where each agent receives the full conversation history on every turn.

Participants:
- Doctor: `gpt-5-nano`
- Psychologist: Ollama `llama3.2`
- Pediatric Doctor: `gpt-4.1-mini` (GPT-4 family)
- Test Subject (child persona): random GPT `gpt-4o-mini`

> Educational simulation only. Not a medical diagnosis or treatment plan.


In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

load_dotenv(override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")

if openai_api_key:
    print(f"OpenAI API key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API key is missing. Add OPENAI_API_KEY to your .env file.")

openai = OpenAI()
ollama = OpenAI(api_key="ollama", base_url="http://localhost:11434/v1")


In [ ]:
participants = [
    {
        "name": "Dr. Amina (General Doctor)",
        "model": "gpt-5-nano",
        "client": "openai",
        "system": """You are a licensed general doctor.
Speak clearly for parents of autistic children.
Use evidence-based recommendations, stay compassionate, and avoid stigma.
Do not prescribe medications in detail; suggest professional follow-up when needed.""",
    },
    {
        "name": "Dr. Leo (Child Psychologist)",
        "model": "llama3.2",
        "client": "ollama",
        "system": """You are a child psychologist using trauma-informed, neurodiversity-affirming language.
Focus on emotional regulation, communication, routine support, and family coaching.""",
    },
    {
        "name": "Dr. Nia (Pediatric Doctor)",
        "model": "gpt-4.1-mini",
        "client": "openai",
        "system": """You are a pediatric doctor in the GPT-4 family.
Prioritize early developmental support, referrals, and practical parent guidance.""",
    },
    {
        "name": "Sam (Test Subject, 5-year-old child persona)",
        "model": "gpt-4o-mini",
        "client": "openai",
        "system": """You are role-playing a 5-year-old autistic child for simulation only.
Use very short, simple sentences and describe feelings/sensory experiences.
Do not give medical advice.""",
    },
]


In [ ]:
def format_history(history):
    return "\n".join([f"{turn['speaker']}: {turn['message']}" for turn in history])


def call_model(client_name, model, messages):
    client = openai if client_name == "openai" else ollama
    kwargs = {}
    if model.startswith("gpt-5"):
        kwargs["reasoning_effort"] = "minimal"
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        **kwargs
    )
    return response.choices[0].message.content.strip()


def one_turn(participant, history):
    conversation_so_far = format_history(history)
    user_prompt = f"""
You are now responding as {participant['name']}.

Topic: autism in a 5-year-old child and the best supports/treatments.

Conversation so far:
{conversation_so_far}

Response rules:
- Keep your answer to 3-5 sentences.
- Be non-stigmatizing and practical.
- Prefer evidence-based supports like early intervention, speech therapy, occupational therapy, parent coaching, and school accommodations.
- Mention that treatment should be personalized.
"""

    messages = [
        {"role": "system", "content": participant["system"]},
        {"role": "user", "content": user_prompt},
    ]

    return call_model(participant["client"], participant["model"], messages)


In [ ]:
conversation_history = [
    {
        "speaker": "Parent",
        "message": "My child is 5 years old with autism. What are the best treatments to help communication, behavior, and sleep?"
    }
]

display(Markdown(f"### {conversation_history[0]['speaker']}\n{conversation_history[0]['message']}"))


In [ ]:
rounds = 3

for _ in range(rounds):
    for participant in participants:
        reply = one_turn(participant, conversation_history)
        conversation_history.append({"speaker": participant["name"], "message": reply})
        display(Markdown(f"### {participant['name']} ({participant['model']})\n{reply}"))

print("Conversation complete.")


In [ ]:
# Optional: inspect full transcript at the end
for turn in conversation_history:
    print(f"{turn['speaker']}: {turn['message']}\n")
